# GPU sharing lab: time-slicing and MPS

This notebook lets you see, on a real GPU, how jobs share it under **time-slicing** and under **MPS** (Multi-Process Service), with and without limits.

| Lab | Question | Typical GPU time |
|---|---|---|
| 1. Time-slicing | How do jobs share compute by default? | about 1 min |
| 2. Time-slicing memory | Is one job's memory protected from another? | about 1 min |
| 3. MPS | Do small jobs run better together under MPS? | about 2 min |
| 4. MPS limits | Can we cap a job's compute and memory? | about 2 min |
| 5. Use case | Can a fast service share a GPU with a heavy job? | about 2 min |

**How it works.** Each "job" is a small Python program (`gpu_worker.py`) that does matrix maths on the GPU and records when each step finishes. The notebook starts several jobs at once, waits for them, and turns their timings into charts.

**Before you start**
- Run on the lab A100 with **MIG turned off** and **no other jobs on the GPU**. Other work on the same GPU changes the results.
- Only one person should run the labs on a given GPU at a time.
- Run the cells **in order**, top to bottom. No root access is needed: the notebook starts and stops its own MPS daemon.
- Results vary between runs and GPUs. Look at the **shape** of each chart, not the exact numbers.

## Setup
Settings you may change are in the first cell. `QUICK_RUN = True` shortens every lab for a fast dry run.

In [ ]:
import os, sys, json, time, shutil, subprocess, threading
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ---- Settings you may change ----------------------------------------------
GPU_INDEX = 0                      # which GPU to use (nvidia-smi index)
QUICK_RUN = os.environ.get("GPU_LAB_QUICK", "0") == "1"   # True = short dry run
BIG = 8192                         # matrix size of a "heavy" job (fills the GPU)
SMALL = 1024                       # matrix size of a "small" job (leaves most of the GPU idle)
SMALL_REPS = 20                    # small multiplications per step, so each step is measurable
# ---------------------------------------------------------------------------

STEP = 2 if QUICK_RUN else 6       # seconds between jobs joining in Labs 1 and 3
RUN = 4 if QUICK_RUN else 10       # seconds per measurement in Labs 4 and 5
LAB_DIR = Path.cwd() / "gpu_lab_runs"
WORKER = LAB_DIR / "gpu_worker.py"
MPS_PIPE = LAB_DIR / "mps-pipe"
MPS_LOG = LAB_DIR / "mps-log"
LAB_DIR.mkdir(exist_ok=True)

COL = {"dark": "#14532D", "green": "#1E7B46", "light": "#A8D5B5", "grey": "#6B7280", "amber": "#E0962A", "red": "#C93C3C"}
JOB_COLOURS = [COL["dark"], COL["green"], COL["grey"], COL["amber"], COL["light"], "#4B5563"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 12, "font.size": 10})
SUMMARY = {}                       # key results, collected for the final dashboard
print("Quick run" if QUICK_RUN else "Full run", "| results folder:", LAB_DIR)

### Check the GPU
This confirms the GPU is free, MIG is off, and PyTorch can see the GPU. It does not keep the GPU busy.

In [ ]:
def sh(cmd, env=None, inp=None, timeout=60):
    """Run a shell command and return its output as text."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env, input=inp, timeout=timeout)
    return (r.stdout + r.stderr).strip()

def gpu_query(fields):
    out = sh(f"nvidia-smi -i {GPU_INDEX} --query-gpu={','.join(fields)} --format=csv,noheader,nounits")
    return dict(zip(fields, [v.strip() for v in out.split(",")]))

info = gpu_query(["name", "memory.total", "memory.used", "mig.mode.current", "compute_mode"])
GPU_MEM_GB = float(info["memory.total"]) / 1024
apps = sh(f"nvidia-smi -i {GPU_INDEX} --query-compute-apps=pid,process_name,used_memory --format=csv,noheader")
torch_check = sh(f'"{sys.executable}" -c "import torch; print(torch.__version__, torch.cuda.is_available())"')

display(pd.DataFrame({"Check": ["GPU", "Memory", "MIG mode", "Compute mode", "Jobs already on the GPU", "PyTorch (version, sees GPU)"],
                      "Result": [info["name"], f"{GPU_MEM_GB:.0f} GB", info["mig.mode.current"], info["compute_mode"],
                                 apps if apps else "none", torch_check]}))
problems = []
if info["mig.mode.current"].lower() == "enabled": problems.append("MIG is enabled: turn it off with `sudo nvidia-smi -i %d -mig 0`." % GPU_INDEX)
if "exclusive" in info["compute_mode"].lower(): problems.append("Compute mode is exclusive: set it back with `sudo nvidia-smi -i %d -c DEFAULT`." % GPU_INDEX)
if apps: problems.append("Other jobs are using this GPU. Stop them, or results will be distorted.")
if not torch_check.endswith("True"): problems.append("PyTorch cannot see the GPU from this Python environment.")
display(Markdown("**Ready to go.**" if not problems else "**Fix before continuing:**\n\n" + "\n".join("- " + p for p in problems)))

### Write the job program
This cell saves `gpu_worker.py`, the program each job runs. You do not need to edit it.

In [ ]:
%%writefile gpu_lab_runs/gpu_worker.py
#!/usr/bin/env python3
"""gpu_worker.py: one GPU job for the GPU sharing lab notebook.

Runs matrix multiplications on the GPU and records when each iteration
(or request) starts and finishes, so the notebook can chart how jobs
share the GPU. Written by the notebook; you do not need to run it by hand.
"""
import argparse
import json
import os
import time

p = argparse.ArgumentParser()
p.add_argument("--tag", required=True, help="job name shown in the charts")
p.add_argument("--out", required=True, help="where to write the results (JSON)")
p.add_argument("--go-file", required=True, help="file the notebook writes when all jobs are ready")
p.add_argument("--offset", type=float, default=0.0, help="seconds after the common start to begin")
p.add_argument("--duration", type=float, default=20.0, help="seconds to run")
p.add_argument("--mode", choices=["throughput", "latency"], default="throughput")
p.add_argument("--size", type=int, default=8192, help="matrix size; larger = heavier job")
p.add_argument("--reps", type=int, default=1, help="matrix multiplications per iteration")
p.add_argument("--hold-gb", type=float, default=0.0, help="extra GPU memory to take at the start")
p.add_argument("--interval-ms", type=float, default=50.0, help="latency mode: time between requests")
a = p.parse_args()

res = {
    "tag": a.tag, "pid": os.getpid(), "mode": a.mode, "size": a.size, "reps": a.reps,
    "hold_gb": a.hold_gb, "offset": a.offset, "duration": a.duration,
    "status": "starting", "message": "", "events": [],
    "under_mps": bool(os.environ.get("CUDA_MPS_PIPE_DIRECTORY")),
    "thread_pct": os.environ.get("CUDA_MPS_ACTIVE_THREAD_PERCENTAGE", ""),
    "mem_limit": os.environ.get("CUDA_MPS_PINNED_DEVICE_MEM_LIMIT", ""),
}


def save():
    tmp = a.out + ".tmp"
    with open(tmp, "w") as f:
        json.dump(res, f)
    os.replace(tmp, a.out)


def mark_ready():
    open(a.out + ".ready", "w").close()


try:
    import torch

    torch.backends.cuda.matmul.allow_tf32 = False
    dev = torch.device("cuda")
    free, total = torch.cuda.mem_get_info()
    res["device"] = torch.cuda.get_device_name(0)
    res["visible_total_gb"] = round(total / 2**30, 2)
    x = torch.randn(a.size, a.size, device=dev)
    y = torch.randn(a.size, a.size, device=dev)
    for _ in range(3):                      # warm-up
        z = x @ y
    torch.cuda.synchronize()
    res["status"] = "ready"
    save()
    mark_ready()

    # Wait for the notebook's common start signal.
    deadline = time.time() + 600
    while not os.path.exists(a.go_file):
        if time.time() > deadline:
            raise TimeoutError("no start signal from the notebook")
        time.sleep(0.01)
    base = float(open(a.go_file).read().strip())
    start_at, stop_at = base + a.offset, base + a.offset + a.duration
    while time.time() < start_at:
        time.sleep(0.002)

    if a.hold_gb > 0:                        # take extra memory, like a large model
        res["alloc_time"] = time.time()
        res["status"] = "allocating"
        hold = torch.empty(int(a.hold_gb * 2**30), dtype=torch.uint8, device=dev)
        hold.fill_(1)
        torch.cuda.synchronize()
        res["alloc_ok"] = True

    res["status"] = "running"
    next_t = time.time()
    while time.time() < stop_at:
        if a.mode == "latency":              # one request every interval
            now = time.time()
            if now < next_t:
                time.sleep(next_t - now)
                continue
        t0 = time.time()
        for _ in range(a.reps):
            z = x @ y
        torch.cuda.synchronize()
        res["events"].append([t0, time.time()])
        next_t = t0 + a.interval_ms / 1000.0
    res["status"] = "ok"
except Exception as e:                       # report out-of-memory and other errors to the notebook
    msg = f"{type(e).__name__}: {e}"
    res["status"] = "oom" if ("out of memory" in msg.lower() or "OutOfMemory" in msg) else "error"
    res["message"] = msg[:400]
    if a.hold_gb > 0 and "alloc_time" in res:
        res["alloc_ok"] = False
finally:
    save()
    try:
        mark_ready()
    except Exception:
        pass

### Helper functions
These start jobs, start and stop MPS, sample GPU memory, and turn timings into rates. Run the cell; reading it is optional.

In [ ]:
def mps_env():
    env = os.environ.copy()
    env["CUDA_MPS_PIPE_DIRECTORY"], env["CUDA_MPS_LOG_DIRECTORY"] = str(MPS_PIPE), str(MPS_LOG)
    env["CUDA_VISIBLE_DEVICES"] = str(GPU_INDEX)          # the daemon serves this one GPU
    return env

def mps_running():
    return (MPS_PIPE / "control").exists() or bool(sh("pgrep -f 'nvidia-cuda-mps-contro[l]' || true"))

def start_mps():
    """Start an MPS control daemon just for this notebook (no root needed)."""
    MPS_PIPE.mkdir(parents=True, exist_ok=True); MPS_LOG.mkdir(parents=True, exist_ok=True)
    if not (MPS_PIPE / "control").exists():
        out = sh("nvidia-cuda-mps-control -d", env=mps_env())
        if out: print(out)
        time.sleep(1)
    print("MPS daemon running:", (MPS_PIPE / "control").exists())

def stop_mps():
    if (MPS_PIPE / "control").exists():
        sh("nvidia-cuda-mps-control", env=mps_env(), inp="quit\n")
        for _ in range(50):
            if not (MPS_PIPE / "control").exists(): break
            time.sleep(0.2)
    print("MPS daemon stopped")

def job_env(mps, thread_pct=None, mem_limit_gb=None):
    env = {k: v for k, v in os.environ.items() if not k.startswith("CUDA_MPS_")}
    if mps:
        env["CUDA_MPS_PIPE_DIRECTORY"], env["CUDA_MPS_LOG_DIRECTORY"] = str(MPS_PIPE), str(MPS_LOG)
        env.pop("CUDA_VISIBLE_DEVICES", None)              # MPS clients should not set it
        if thread_pct: env["CUDA_MPS_ACTIVE_THREAD_PERCENTAGE"] = str(thread_pct)
        if mem_limit_gb: env["CUDA_MPS_PINNED_DEVICE_MEM_LIMIT"] = f"0={mem_limit_gb}G"
    else:
        env["CUDA_VISIBLE_DEVICES"] = str(GPU_INDEX)
    return env

class Run:
    def __init__(self, **kw): self.__dict__.update(kw)

def run_jobs(jobs, mps=False, name="run", sample_memory=False, snapshot_at=None):
    """Start all jobs, let them warm up, then give one common start signal.
    Each job dict: tag, duration, and optionally offset, mode, size, reps, hold_gb,
    interval_ms, thread_pct, mem_limit_gb."""
    run_dir = LAB_DIR / f"{name}-{time.strftime('%H%M%S')}"
    run_dir.mkdir(parents=True, exist_ok=True)
    go = run_dir / "go"
    procs = []
    for j in jobs:
        out = run_dir / f"{j['tag']}.json"
        cmd = [sys.executable, str(WORKER), "--tag", j["tag"], "--out", str(out), "--go-file", str(go),
               "--offset", str(j.get("offset", 0)), "--duration", str(j["duration"]),
               "--mode", j.get("mode", "throughput"), "--size", str(j.get("size", BIG)),
               "--reps", str(j.get("reps", 1)), "--hold-gb", str(j.get("hold_gb", 0)),
               "--interval-ms", str(j.get("interval_ms", 50))]
        log = open(run_dir / f"{j['tag']}.log", "w")
        p = subprocess.Popen(cmd, env=job_env(mps, j.get("thread_pct"), j.get("mem_limit_gb")), stdout=log, stderr=subprocess.STDOUT)
        procs.append((j, out, p, log))
    deadline = time.time() + 180
    while time.time() < deadline:                          # wait until every job is warmed up
        if all(Path(str(o) + ".ready").exists() or p.poll() is not None for _, o, p, _ in procs): break
        time.sleep(0.2)
    base = time.time() + 1.0
    go.write_text(repr(base))
    end = base + max(j.get("offset", 0) + j["duration"] for j in jobs)
    samples, snap = [], None
    def sampler():
        while time.time() < end + 1:
            v = sh(f"nvidia-smi -i {GPU_INDEX} --query-gpu=memory.used --format=csv,noheader,nounits")
            try: samples.append((time.time() - base, float(v.split()[0])))
            except (ValueError, IndexError): pass
            time.sleep(0.5)
    th = threading.Thread(target=sampler, daemon=True) if sample_memory else None
    if th: th.start()
    if snapshot_at is not None:
        time.sleep(max(0, base + snapshot_at - time.time()))
        snap = sh(f"nvidia-smi -i {GPU_INDEX}")
    for j, o, p, log in procs:
        try: p.wait(timeout=max(5, end + 60 - time.time()))
        except subprocess.TimeoutExpired: p.kill()
        log.close()
    if th: th.join(timeout=5)
    results = {}
    for j, o, p, log in procs:
        if o.exists(): results[j["tag"]] = json.loads(o.read_text())
        else: results[j["tag"]] = {"tag": j["tag"], "status": "missing", "events": [],
                                   "message": (run_dir / f"{j['tag']}.log").read_text()[-400:]}
    bad = [r for r in results.values() if r["status"] in ("error", "missing")]
    for r in bad: print(f"Job {r['tag']} did not run: {r['message']}")
    mem = pd.DataFrame(samples, columns=["t", "used_mib"]) if sample_memory else None
    return Run(base=base, jobs=jobs, results=results, memory=mem, snapshot=snap, dir=run_dir)

def rate_series(res, base, t_end, binw):
    """Steps per second over time, spreading each step evenly over how long it took."""
    edges = np.arange(0, t_end + binw, binw)
    rate = np.zeros(len(edges) - 1)
    for s, e in res.get("events", []):
        s, e = s - base, e - base
        d = max(e - s, 1e-9)
        for k in range(len(rate)):
            ov = min(e, edges[k + 1]) - max(s, edges[k])
            if ov > 0: rate[k] += ov / d
    return (edges[:-1] + edges[1:]) / 2, rate / binw

def window_rate(res, base, t0, t1):
    """Average steps per second for steps that finished between t0 and t1."""
    n = sum(1 for s, e in res.get("events", []) if t0 <= e - base <= t1)
    return n / max(t1 - t0, 1e-9)

def step_times_ms(res, base, t0, t1):
    return np.array([(e - s) * 1000 for s, e in res.get("events", []) if t0 <= e - base <= t1])

def staggered_jobs(n, size, reps):
    """n jobs that join one every STEP seconds and all stop together."""
    total = STEP * (n + 1)
    return [{"tag": f"Job {chr(65 + k)}", "offset": k * STEP, "duration": total - k * STEP, "size": size, "reps": reps}
            for k in range(n)]

def phase_table(run, n):
    """Per-job and total speed while 1, 2, ... n jobs were running, as % of one job alone."""
    tags = [j["tag"] for j in run.jobs]
    alone = window_rate(run.results[tags[0]], run.base, 0.5, STEP)
    rows = []
    for k in range(1, n + 1):
        t0, t1 = (k - 1) * STEP + 0.5, k * STEP + (STEP if k == n else 0)
        rates = [window_rate(run.results[t], run.base, t0, t1) for t in tags[:k]]
        rows.append({"Jobs running": k, "Each job (% of alone)": round(100 * np.mean(rates) / alone),
                     "Total (% of alone)": round(100 * sum(rates) / alone)})
    return pd.DataFrame(rows).set_index("Jobs running"), alone

def stacked_chart(ax, run, alone, title, binw=None):
    binw = binw or STEP / 3
    tags = [j["tag"] for j in run.jobs]
    t_end = max(j["offset"] + j["duration"] for j in run.jobs)
    series = []
    for t in tags:
        x, r = rate_series(run.results[t], run.base, t_end, binw)
        series.append(100 * r / alone)
    ax.stackplot(x, series, labels=tags, colors=JOB_COLOURS[:len(tags)], alpha=0.9)
    for k in range(1, len(tags)):
        ax.axvline(k * STEP, color="white", lw=1, ls="--")
    ax.set(title=title, xlabel="Seconds", ylabel="% of one job's speed alone", xlim=(0, t_end))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=9, ncol=len(tags), frameon=False)

print("Helpers loaded")

---
## Lab 1: Time-slicing, how jobs share compute by default

**Scenario.** Four heavy jobs join one GPU one after another. Nothing special is switched on: this is what happens whenever several programs use one GPU.

**What happens under the hood.** Each job has its own *context* on the GPU. The GPU runs one context at a time and switches between them every few milliseconds. So every job gets the whole GPU, but only for its turn.

**What to look for**
- The total height of the chart stays roughly level as jobs join.
- Each new job takes its share from the others: with four jobs, each gets about a quarter.

In [ ]:
lab1 = run_jobs(staggered_jobs(4, BIG, 1), name="lab1")
table1, alone1 = phase_table(lab1, 4)
display(table1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.6, 1]})
stacked_chart(axes[0], lab1, alone1, "Each colour is one job: the total stays roughly level")
a_alone = step_times_ms(lab1.results["Job A"], lab1.base, 0.5, STEP)
a_four = step_times_ms(lab1.results["Job A"], lab1.base, 3 * STEP + 0.5, 5 * STEP)
axes[1].boxplot([a_alone, a_four], patch_artist=True,
                boxprops=dict(facecolor=COL["light"], color=COL["dark"]), medianprops=dict(color=COL["dark"]))
axes[1].set_xticks([1, 2], ["Alone", "With 3 others"])
axes[1].set(title="Job A: time per step", ylabel="Milliseconds")
plt.tight_layout(); plt.show()
SUMMARY["lab1"] = table1

**What this shows.** Time-slicing divides the GPU's **time**, not its power. Adding a job does not add capacity: it slows everyone down. Job A's steps take roughly four times longer once three other jobs share the GPU.

**Try this.** Set `SMALL` jobs instead of `BIG` in `staggered_jobs(4, BIG, 1)`, for example `staggered_jobs(4, SMALL, SMALL_REPS)`. Small jobs leave most of the GPU idle during their turn. Keep the chart in mind for Lab 3.

---
## Lab 2: Time-slicing, is one job's memory protected?

**Scenario.** Job A loads a large model and takes about 60% of GPU memory. A few seconds later, Job B starts and asks for about 50%.

**What happens under the hood.** Under time-slicing, compute is taken in turns, but **memory is one open pool**. There is no per-job limit: whoever allocates first keeps it.

**What to look for**
- GPU memory in use jumps when Job A allocates.
- Job B's request fails with *out of memory*, while Job A carries on unaffected.

In [ ]:
hold_a, hold_b = round(0.60 * GPU_MEM_GB, 1), round(0.50 * GPU_MEM_GB, 1)
b_at = STEP + 1
lab2 = run_jobs([{"tag": "Job A", "offset": 1, "duration": 3 * STEP + 2, "hold_gb": hold_a},
                 {"tag": "Job B", "offset": b_at, "duration": STEP, "hold_gb": hold_b}],
                name="lab2", sample_memory=True)
rows = []
for tag, want in [("Job A", hold_a), ("Job B", hold_b)]:
    r = lab2.results[tag]
    rows.append({"Job": tag, "Asked for (GB)": want, "Outcome": {"ok": "Ran normally", "oom": "Out of memory"}.get(r["status"], r["status"]),
                 "Steps completed": len(r["events"]), "Message": r["message"][:80]})
display(pd.DataFrame(rows).set_index("Job"))

m = lab2.memory
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(m["t"], 100 * m["used_mib"] / (GPU_MEM_GB * 1024), color=COL["green"], lw=2.5)
ax.axhline(100, color=COL["grey"], ls=":", lw=1); ax.text(0.2, 101.5, "Total GPU memory", color=COL["grey"], fontsize=9)
ax.axvline(b_at, color=COL["red"], ls="--")
ok_b = lab2.results["Job B"]["status"] == "ok"
ax.text(b_at + 0.2, 20, f"Job B asks for {hold_b:.0f} GB:\n" + ("it fits" if ok_b else "out of memory"), color=COL["red"], fontweight="bold")
ax.set(title="GPU memory in use", xlabel="Seconds", ylabel="% of GPU memory", ylim=(0, 110))
plt.tight_layout(); plt.show()
SUMMARY["lab2"] = rows

**What this shows.** Time-slicing gives **no memory protection**. A job that arrives later can fail because an earlier job took the memory, even though the GPU's compute has room to spare. In practice, teams agree memory limits per user or framework, or use MPS limits, MIG or vGPU when protection matters.

**Try this.** Swap the order: give Job B `offset: 1` and Job A `offset: b_at`. The job that allocates first always wins.

---
## Lab 3: MPS, do small jobs run better together?

**Scenario.** Four *small* jobs join one at a time. We run the same experiment twice: first with ordinary time-slicing, then with MPS switched on.

**What happens under the hood.** A small job uses only part of the GPU during its turn. With time-slicing, other jobs cannot use the idle part, because work from different contexts does not run side by side. With **MPS**, all jobs send their work through one MPS server, so their work can run **at the same time** on different parts of the GPU.

**What to look for**
- Time-slicing: the total stays roughly level as jobs join.
- MPS: the total usually **rises** as jobs join. How much depends on the job size and the GPU.
- In the `nvidia-smi` snapshot, MPS jobs show the process type **M+C**, and an `nvidia-cuda-mps-server` process appears.

In [ ]:
lab3_ts = run_jobs(staggered_jobs(4, SMALL, SMALL_REPS), name="lab3-timeslice")
start_mps()
try:
    lab3_mps = run_jobs(staggered_jobs(4, SMALL, SMALL_REPS), mps=True, name="lab3-mps", snapshot_at=3.5 * STEP)
finally:
    stop_mps()

t_ts, alone_ts = phase_table(lab3_ts, 4)
t_mps, alone_mps = phase_table(lab3_mps, 4)
both = pd.DataFrame({"Time-slicing: total (%)": t_ts["Total (% of alone)"], "MPS: total (%)": t_mps["Total (% of alone)"]})
display(both)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [1.3, 1.3, 1]})
stacked_chart(axes[0], lab3_ts, alone_ts, "Time-slicing: small jobs take turns")
stacked_chart(axes[1], lab3_mps, alone_mps, "MPS: small jobs run side by side")
top = max(axes[0].get_ylim()[1], axes[1].get_ylim()[1]); axes[0].set_ylim(0, top); axes[1].set_ylim(0, top)
xk = np.arange(1, 5)
axes[2].plot(xk, both.iloc[:, 0], "o-", color=COL["grey"], lw=2.5, label="Time-slicing")
axes[2].plot(xk, both.iloc[:, 1], "o-", color=COL["green"], lw=2.5, label="MPS")
axes[2].set(title="Total speed as jobs join", xlabel="Jobs running", ylabel="% of one job alone", xticks=xk)
axes[2].set_ylim(bottom=0); axes[2].legend(frameon=False)
plt.tight_layout(); plt.show()
SUMMARY["lab3"] = both

print("nvidia-smi while the MPS jobs were running (look for M+C and nvidia-cuda-mps-server):\n")
print("\n".join(l for l in (lab3_mps.snapshot or "").splitlines() if ("M+C" in l or "mps" in l.lower() or "PID" in l)) or lab3_mps.snapshot)

**What this shows.** When jobs are small, time-slicing leaves the GPU partly idle. MPS fills those gaps, so the same GPU completes more work in total. This is why MPS suits many small inference or simulation jobs from **one team**.

**The catch.** All MPS clients share one server, so a serious fault in one job can affect the others. MPS is a performance tool, not a security boundary.

**Try this.** Re-run with `BIG` jobs instead of small ones. Heavy jobs already fill the GPU, so MPS gains little.

---
## Lab 4: MPS limits, capping compute and memory

MPS can put a cap on each job, set through environment variables when the job starts:
- `CUDA_MPS_ACTIVE_THREAD_PERCENTAGE`: the share of the GPU's compute a job may use.
- `CUDA_MPS_PINNED_DEVICE_MEM_LIMIT`: the most GPU memory a job may allocate.

### 4a. Compute cap
**Scenario.** One heavy job runs three times: capped at 100%, 50% and 25% of the GPU.

**What to look for.** The job's speed drops roughly in line with the cap. The match is not exact.

In [ ]:
caps = [100, 50, 25]
start_mps()
try:
    lab4a = run_jobs([{"tag": f"Cap {c}%", "offset": i * RUN, "duration": RUN, "thread_pct": c} for i, c in enumerate(caps)],
                     mps=True, name="lab4a")
finally:
    stop_mps()
speeds = [window_rate(lab4a.results[f"Cap {c}%"], lab4a.base, i * RUN + 0.5, (i + 1) * RUN) for i, c in enumerate(caps)]
rel = [100 * s / speeds[0] for s in speeds]
display(pd.DataFrame({"Cap (% of GPU)": caps, "Speed (% of uncapped)": np.round(rel)}).set_index("Cap (% of GPU)"))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar([f"{c}% cap" for c in caps], rel, color=[COL["dark"], COL["green"], COL["light"]])
ax.plot(range(3), caps, "o--", color=COL["amber"], label="Speed if it matched the cap exactly")
ax.set(title="One heavy job under different MPS compute caps", ylabel="% of uncapped speed", ylim=(0, 115))
ax.legend(frameon=False); plt.tight_layout(); plt.show()
SUMMARY["lab4a"] = dict(zip(caps, rel))

### 4b. Protecting small jobs from a heavy one
**Scenario.** One heavy job and three small jobs share the GPU under MPS. First with no caps, then with the heavy job capped at 25%.

**What to look for.** With the cap, the small jobs usually speed up, and the heavy job slows down. The cap moves capacity from the heavy job to the others.

In [ ]:
def mixed(cap=None):
    jobs = [{"tag": "Heavy", "duration": RUN, "size": BIG, "thread_pct": cap}]
    jobs += [{"tag": f"Small {k}", "duration": RUN, "size": SMALL, "reps": SMALL_REPS} for k in (1, 2, 3)]
    return jobs
start_mps()
try:
    lab4b_open = run_jobs(mixed(), mps=True, name="lab4b-nocap")
    lab4b_cap = run_jobs(mixed(25), mps=True, name="lab4b-cap")
finally:
    stop_mps()
tags = ["Heavy", "Small 1", "Small 2", "Small 3"]
r_open = [window_rate(lab4b_open.results[t], lab4b_open.base, 0.5, RUN) for t in tags]
r_cap = [window_rate(lab4b_cap.results[t], lab4b_cap.base, 0.5, RUN) for t in tags]
change = pd.DataFrame({"No cap (steps/s)": np.round(r_open, 1), "Heavy capped at 25% (steps/s)": np.round(r_cap, 1),
                       "Change": [f"{100 * (c / o - 1):+.0f}%" if o else "n/a" for o, c in zip(r_open, r_cap)]}, index=tags)
display(change)

fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(len(tags)); w = 0.38
ax.bar(x - w / 2, [100] * 4, w, color=COL["grey"], label="No cap")
ax.bar(x + w / 2, [100 * c / o if o else 0 for o, c in zip(r_open, r_cap)], w, color=COL["green"], label="Heavy job capped at 25%")
ax.set(xticks=x, xticklabels=tags, ylabel="% of speed with no cap", title="Effect of capping the heavy job")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False); plt.tight_layout(); plt.show()
SUMMARY["lab4b"] = change

### 4c. Memory cap
**Scenario.** Three jobs run under MPS, each limited to about a quarter of GPU memory. One asks for less than its limit, one asks for more, and one asks for nothing extra.

**What to look for.** Only the job that asks for more than its limit fails. The others carry on normally.

In [ ]:
limit = max(1, int(0.25 * GPU_MEM_GB))
start_mps()
try:
    lab4c = run_jobs([{"tag": "Within limit", "duration": RUN, "hold_gb": round(0.5 * limit, 1), "mem_limit_gb": limit},
                      {"tag": "Over limit", "duration": RUN, "hold_gb": round(1.5 * limit, 1), "mem_limit_gb": limit},
                      {"tag": "No extra memory", "duration": RUN, "mem_limit_gb": limit}], mps=True, name="lab4c")
finally:
    stop_mps()
rows = []
for j in lab4c.jobs:
    r = lab4c.results[j["tag"]]
    rows.append({"Job": j["tag"], "Limit (GB)": limit, "Asked for (GB)": j.get("hold_gb", 0),
                 "Outcome": {"ok": "Ran normally", "oom": "Out of memory"}.get(r["status"], r["status"]),
                 "Steps completed": len(r["events"])})
display(pd.DataFrame(rows).set_index("Job"))
SUMMARY["lab4c"] = rows

**What this shows.** MPS limits stop one job from **starving** the others of compute or memory. They are enforced in software by the MPS server, so they protect fairness, not security: a serious fault in one job can still affect the others.

---
## Lab 5: Use case, a fast service next to a heavy job

**Scenario.** A *service* job answers one small request every 50 ms, like an inference API. We measure how long each request takes:
1. **Alone** on the GPU.
2. Next to a heavy training-style job, with **time-slicing**.
3. Next to the heavy job, with **MPS** and no caps.
4. Next to the heavy job, with **MPS** and the heavy job **capped at 50%**.

**What to look for**
- Sharing makes the slowest requests (the 95th percentile) much slower than running alone.
- Capping the heavy job usually brings response times back down, at the cost of the heavy job's speed.
- The exact ordering of conditions 2 and 3 can vary between GPUs and runs.

In [ ]:
SERVICE = {"tag": "Service", "mode": "latency", "size": 4096, "reps": 1, "interval_ms": 50, "offset": 1, "duration": RUN}
def heavy(cap=None): return {"tag": "Heavy", "duration": RUN + 2, "size": BIG, "thread_pct": cap}

lab5 = {}
lab5["Alone"] = run_jobs([SERVICE], name="lab5-alone")
lab5["Time-slicing"] = run_jobs([SERVICE, heavy()], name="lab5-ts")
start_mps()
try:
    lab5["MPS, no cap"] = run_jobs([SERVICE, heavy()], mps=True, name="lab5-mps")
    lab5["MPS, heavy capped 50%"] = run_jobs([SERVICE, heavy(50)], mps=True, name="lab5-mpscap")
finally:
    stop_mps()

lat = {k: step_times_ms(r.results["Service"], r.base, 1.5, RUN + 1) for k, r in lab5.items()}
heavy_speed = {k: (window_rate(r.results["Heavy"], r.base, 1.5, RUN + 1) if "Heavy" in r.results else np.nan) for k, r in lab5.items()}
base_heavy = heavy_speed["Time-slicing"]
tbl = pd.DataFrame({"Typical request (ms, p50)": [np.percentile(v, 50) for v in lat.values()],
                    "Slow requests (ms, p95)": [np.percentile(v, 95) for v in lat.values()],
                    "Heavy job speed (% of time-slicing)": [100 * heavy_speed[k] / base_heavy if base_heavy else np.nan for k in lat]},
                   index=list(lat)).round(1)
display(tbl)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].boxplot(list(lat.values()), patch_artist=True, showfliers=False,
                boxprops=dict(facecolor=COL["light"], color=COL["dark"]), medianprops=dict(color=COL["dark"]))
axes[0].set_xticks(range(1, len(lat) + 1), [k.replace(", ", ",\n") for k in lat])
axes[0].set(title="Service response time per request", ylabel="Milliseconds")
axes[1].bar([k.replace(", ", ",\n") for k in lat], tbl["Slow requests (ms, p95)"], color=[COL["dark"], COL["grey"], COL["amber"], COL["green"]])
axes[1].set(title="Slowest 5% of requests (p95)", ylabel="Milliseconds")
plt.tight_layout(); plt.show()
SUMMARY["lab5"] = tbl

**What this shows.** A latency-sensitive service suffers most from sharing, because it waits whenever another job holds the GPU. MPS with a cap on the heavy job is a practical middle ground on one GPU. When the service needs a **guaranteed** response time, MIG (or MIG-backed vGPU) is the stronger answer, because it gives the service its own hardware.

---
## Summary: everything on one page

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
t1 = SUMMARY["lab1"]
axes[0, 0].bar(t1.index.astype(str), t1["Each job (% of alone)"], color=COL["grey"], label="Each job")
axes[0, 0].plot(t1.index.astype(str), t1["Total (% of alone)"], "o-", color=COL["dark"], lw=2.5, label="Total")
axes[0, 0].set(title="Lab 1  Time-slicing: total stays level, each job shrinks", xlabel="Jobs running", ylabel="% of one job alone"); axes[0, 0].legend(frameon=False)
b = SUMMARY["lab3"]
axes[0, 1].plot(b.index, b.iloc[:, 0], "o-", color=COL["grey"], lw=2.5, label="Time-slicing")
axes[0, 1].plot(b.index, b.iloc[:, 1], "o-", color=COL["green"], lw=2.5, label="MPS")
axes[0, 1].set(title="Lab 3  Small jobs: MPS raises the total", xlabel="Jobs running", ylabel="% of one job alone", xticks=list(b.index)); axes[0, 1].set_ylim(bottom=0); axes[0, 1].legend(frameon=False)
c = SUMMARY["lab4a"]
axes[1, 0].bar([f"{k}%" for k in c], list(c.values()), color=[COL["dark"], COL["green"], COL["light"]])
axes[1, 0].set(title="Lab 4  MPS compute cap: speed follows the cap", xlabel="Cap", ylabel="% of uncapped speed", ylim=(0, 115))
t5 = SUMMARY["lab5"]
axes[1, 1].bar([k.replace(", ", ",\n") for k in t5.index], t5["Slow requests (ms, p95)"], color=[COL["dark"], COL["grey"], COL["amber"], COL["green"]])
axes[1, 1].set(title="Lab 5  Service p95 response time", ylabel="Milliseconds")
plt.tight_layout(); plt.show()

| What we saw | Time-slicing | MPS, no limits | MPS, with limits |
|---|---|---|---|
| Jobs run | One at a time, in turns | Side by side | Side by side, within caps |
| Small jobs | GPU partly idle | More total work done | More total work, fairly shared |
| Memory | One open pool; first come, first served | One open pool | Capped per job |
| A heavy neighbour | Slows everyone | Can crowd others out | Held to its cap |
| Isolation | None | None; shared fault risk | Fairness, not security |

**Discussion questions**
1. Which of your team's workloads look like Lab 1, and which look like Lab 3?
2. In Lab 5, what response time would your service need, and would MPS with caps meet it or would you need MIG?
3. What memory rules would you need if you used time-slicing for notebooks?

### Clean up
Stops MPS if it is still running and removes the run folders. Run this when you have finished.

In [ ]:
stop_mps()
leftover = sh("pgrep -f 'gpu_lab_runs/gpu_worke[r].py' || true")
if leftover:
    sh(f"kill {' '.join(leftover.split())}")
    print("Stopped leftover jobs:", leftover.split())
for d in LAB_DIR.glob("*-*"):
    if d.is_dir() and d.name not in ("mps-pipe", "mps-log"):
        shutil.rmtree(d, ignore_errors=True)
print("Clean. GPU state:", gpu_query(["memory.used", "compute_mode"]))

### Further reading
- NVIDIA Multi-Process Service documentation: https://docs.nvidia.com/deploy/mps/index.html
- NVIDIA GPU Operator, time-slicing GPUs in Kubernetes: https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/gpu-sharing.html
- NVIDIA MIG User Guide: https://docs.nvidia.com/datacenter/tesla/mig-user-guide/latest/